# 🏃 Merlino + Gemma 4 — Semantic Sovereignty with AOOE

**Connected Data · hands-on workshop · fictional athlete Alex**

> **Own the meanings. Govern the decisions. Choose the AI.**

**Traditional ontology engineering** supplies a Legend (identity and vocabulary) and Map (things/relationships). **Agent-Oriented Ontology Engineering** adds a Rulebook: which actions are allowed, which evidence they require, independent SHACL validation and an auditable trace.

This notebook builds on Bea's *Merlin Running Pal* Map and Railroad. The new engine fixes action/approval side effects before validation and adds **optional real Gemma 4 function calling**. An LLM may propose a week; it **cannot approve it or rewrite the ontology**.

### Important data and safety boundaries

- All athlete records are **synthetic**. Do not upload real health or workout histories to this workshop notebook.
- Colab runs on **Google-managed infrastructure**. Running model inference in the Colab VM does **not** mean the computation is on your personal machine or that Google does not operate the host.
- Gemma is **off by default until enabled explicitly** in the model cell; running the other cells executes real RDF/SHACL validation but is **not** a generative AI demonstration.
- These progression thresholds are **fictional for teaching**, not medical or coaching recommendations.
- An out-of-process PySHACL call is a separate validator step, not an enterprise security boundary without protected access control.


## 0 · Install and unpack the reviewed educational source bundle

The source below is embedded in this notebook so practitioners never need access to private WoCell, methodology or WoGraph repositories. Python packages are installed from PyPI. A failed install stops the workshop rather than claiming validation succeeded.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
                       'rdflib>=7.1,<9', 'pyshacl>=0.30,<1',
                       'requests>=2.31,<3', 'pytest>=8,<10'])
print('RDFLib, genuine PySHACL, and pytest installed for Python', sys.version.split()[0])


In [ ]:
import base64, io, zipfile
from pathlib import Path
PROJECT = Path.cwd() / 'merlino_gemma_workshop'
PROJECT.mkdir(exist_ok=True)
EMBEDDED = (
    '...'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(''.join(EMBEDDED)))) as bundle:
    # Trusted notebook-internal reviewed archive; no arbitrary path from model input.
    for member in bundle.infolist():
        path = PROJECT / member.filename
        if not path.resolve().is_relative_to(PROJECT.resolve()):
            raise ValueError('Invalid embedded archive path')
    bundle.extractall(PROJECT)
sys.path.insert(0, str(PROJECT))
print('Reviewable source extracted to:', PROJECT)


## 1 · Legend + Map: Bea's ontology and meanings

Observe that *Tempo Run* has an **explicit classification** (`mr:isHard true`). That is an organizationally owned meaning—not a prediction from the model. The agent's Railroad is separate from the domain Map.

In [ ]:
from rdflib import Graph
from merlino.engine import Merlino, MR, MG, example_plan
from merlino.competency import run_checks
from pprint import pprint

ontology_demo = Merlino()
print('Ontology triples:', len(ontology_demo.map))
print('TempoRun isHard:', ontology_demo.map.value(MR.TempoRun, MR.isHard))
print('EasyRun  isHard:', ontology_demo.map.value(MR.EasyRun, MR.isHard))
print('Training data and controlled vocabulary:')
for row in ontology_demo.map.query("""SELECT ?run ?type WHERE {
  ?run <https://example.org/merlin-railroad#workoutType> ?type
} LIMIT 8"""):
    print(str(row.run).rsplit('/',1)[-1], '→', str(row.type).rsplit('#',1)[-1])

### Evidence, not a prompt constant

Bea's Map models eight synthetic runs (31 km, then 33 km) as `schema:ExerciseAction` with a numeric `mr:distanceKm`. The age is **reference date − latest run**. The fixture is anchored to 27 October 2026 to make the exercises reproducible; using today's actual date will make these old records stale, as it should. **Only use synthetic CSV data in hosted Colab.**


In [ ]:
from merlino.history import history_summary, load_synthetic_csv
from merlino.history import SCHEMA
summary = history_summary(ontology_demo.map)
print('Graph-derived last week (km):', summary['baseline_km'])
print('Last run:', summary['last_run_date'], '| reference date:', summary['as_of_date'])
print('Age (days):', summary['age_days'])
assert summary['baseline_km'] == 33 and summary['age_days'] == 2
print('Example SPARQL — source observations:')
for row in ontology_demo.map.query("""PREFIX schema: <https://schema.org/>
PREFIX mr: <https://example.org/merlin-railroad#>
SELECT ?run ?t ?km WHERE { ?run a schema:ExerciseAction ; schema:startTime ?t ; mr:distanceKm ?km }
ORDER BY ?t"""):
    print(str(row.run).split('/')[-1], row.t, row.km)


## 2 · Railroad: inspect Bea's five agent competency questions

These are **design-time graph queries**, not a certification or proof of medical safety. Empty counterexample sets mean these five specific checks pass.

In [ ]:
for name, (question, counterexamples) in run_checks(ontology_demo.railroad).items():
    print(('PASS' if not counterexamples else 'FAIL'), name,
          '| counterexamples:', counterexamples)

### From map to railroad: an agent's authority is data

Bea's Mermaid-like railroad is a real RDF graph: nine states, ten actions, two actor roles and three evidence classes. The five competency questions test the **state space before any agent runs** (they are checks on this graph, not proofs of universal correctness).


In [ ]:
from merlino.engine import OPS
from rdflib.namespace import RDF
states = set(ontology_demo.railroad.subjects(RDF.type, OPS.State))
actions = set(ontology_demo.railroad.subjects(RDF.type, OPS.Action))
roles = set(ontology_demo.railroad.subjects(RDF.type, OPS.AgentRole))
print('States:', len(states), 'Actions:', len(actions), 'Roles:', len(roles))
assert (len(states),len(actions),len(roles)) == (9,10,2)
print('The next permitted steps are derived from the railroad, not added to the prompt by hand.')


## 3 · Rulebook: independent SHACL gate **before** writing

The runner's baseline is **33 km/week**. The fictional v1 policy allows at most **1.10 ×** baseline. The candidate for 42.5 km is rejected; the engine must still hold **no plan**. The 34.5 km candidate is permitted as a *proposal*, but not approved or scheduled.

In [ ]:
engine = Merlino()  # Real external subprocess gate; no injected stub
baseline = engine.read_history()
assert baseline.accepted, baseline.violations  # fail closed if PySHACL is missing
print('Read history:', baseline.as_dict())

blocked = engine.propose(42.5)
print('\n42.5 km candidate:'); pprint(blocked.as_dict())
assert not blocked.accepted and engine.plan is None, 'Blocked action MUST NOT mutate the world'

eligible = engine.propose(34.5)
print('\n34.5 km candidate:'); pprint(eligible.as_dict())
assert eligible.accepted and engine.approval is None and not engine.started

## 4 · Bea's seven scenarios: two conditions, one set of mistakes

Run the same deterministic intentions in (a) a **disposable no-rules shadow** and (b) the **independently governed** Merlino. Both call real PySHACL. The shadow intentionally allows invalid actions **only on synthetic in-memory state** so that we can measure what would pass without the rules. It is not a deployment mode, and it never calls external services. The real Gemma model is **not** being sampled here.

S1 normal; S2 42.5 km overload; S3 21-day-old history; S4 pain; S5 changed plan after approval; S6 consecutive hard days; S7 no runner approval.


In [ ]:
from merlino.scenarios import run_all, SCENARIOS
ROWS, EPISODES = run_all()  # invokes real SHACL; fail closed if unavailable
for case_id, title in SCENARIOS:
    a = next(x for x in ROWS if x['scenario']==case_id and x['condition'].startswith('no rules'))
    b = next(x for x in ROWS if x['scenario']==case_id and x['condition']=='railroad + SHACL')
    print(case_id, title)
    print('  NO RULES (shadow):',a['weeks_started'],'km weeks; broken:',a['broken_rules_executed'])
    print('  GOVERNED:',b['weeks_started'],'km weeks; blocked:',b['blocked_by'],
          'warned:',len(b['warned']), 'end:',b['end_state'])
print('All 14 deterministic episodes executed against the same AOOE engine.')


### Replay S5 — three independent reasons to block

Merlino proposes 34 km, the runner approves, then an edited 38 km version attempts to start **without re-proposal or re-approval**. B2 checks the **sum of session kilometres**, B4 checks the exact approval digest and policy/meaning revisions, and B7 requires a previously accepted proposal. The model cannot waive any of them.


In [ ]:
from merlino.engine import TRACE, PROV
s5 = next(e for e in EPISODES if e.scenario=='S5' and e.condition=='railroad_rules')
for step in s5.steps:
    print(f"{step['step']:02d}",step['action'],'EXECUTED' if step['executed'] else 'BLOCKED',
          ','.join(step['codes']) or 'ok')
assert {'B2','B4','B7'} <= set(s5.summary()['blocked_by'])
print('RDF PROV-O audit events:',len(s5.engine.provenance))
WHY = """PREFIX mg: <https://example.org/merlino-gemma#>
PREFIX prov: <http://www.w3.org/ns/prov#>
SELECT ?decision ?action ?policy ?meaning ?error WHERE {
  ?decision a prov:Activity ; mg:actionName ?action ; mg:policyVersion ?policy ;
            mg:meaningRevision ?meaning .
  OPTIONAL { ?decision mg:violation ?error }
} ORDER BY ?decision"""
for row in s5.engine.provenance.query(WHY):
    if row.error is not None: print(str(row.decision).split('/')[-1], row.action, row.error)


### Replay S4 — why the latest check-in must be a context fact

The gate sees a **host-constructed latest check-in** and a persistent pain hold. The agent cannot attach an older pain-free record to remove a more recent pain report. Trying to request another check-in after pain fails. Only human handover proceeds.


In [ ]:
s4 = next(e for e in EPISODES if e.scenario=='S4' and e.condition=='railroad_rules')
print('Latest confirmed check-in:',s4.engine.latest_checkin)
print('After pain, blocked rules:',s4.summary()['blocked_by'])
print('Final state:',s4.engine.state_name)
assert 'B3' in s4.summary()['blocked_by'] and s4.engine.state_name=='HandedToHuman'


### Regression tests — turn every hole into a test

The original Merlin Running Pal notebook reported **23/23 passing checks** on its own deterministic implementation. This port has a **different** test suite (including all seven paired scenarios, the five design checks and attacks against approval, pain context, and distance sums). We do not describe its test count as Bea's original 23. The notebook will fail if real PySHACL validation does not pass.


In [ ]:
import subprocess, sys
from pathlib import Path

suite = ['tests/test_engine.py', 'tests/test_workshop_cases.py',
         'tests/test_memory_and_bea_feedback.py', 'tests/test_episodic_experiment.py']
print('Python:', sys.version.split()[0], '| Executable:', sys.executable)
print('Project:', Path(PROJECT).resolve(), '| Test files present:',
      all((Path(PROJECT) / name).is_file() for name in suite))
result = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-ra', *suite],
                        cwd=str(PROJECT), text=True, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT)
print(result.stdout, flush=True)
if result.returncode:
    raise RuntimeError(f'Merlino test suite failed (pytest exit code {result.returncode}); '
                       'read the FAILED / ERROR lines printed above this traceback')


## 5 · Semantic Sovereignty exercise: edit a meaning, not a prompt

Bea's scenario has Wednesday Intervals and Thursday TempoRun. SHACL's **B5 warning** depends on `mr:isHard` in the Map. A steward deliberately changes that classification, invalidating any previous approval and changing the validation result. **In real training, such a definition change would require expert governance.**

In [ ]:
semantic = Merlino()
assert semantic.read_history().accepted
original = semantic.propose(34.5, style='consecutive_hard')
print('Before meaning edit:', original.warnings)
assert any('B5' in warning for warning in original.warnings)
old_revision = semantic.meaning_revision

semantic.revise_tempo_meaning(tempo_is_hard=False, steward_confirmed=True)
changed = semantic.propose(34.5, style='consecutive_hard')
print('After steward-approved meaning edit:', changed.warnings)
print('Ontology revision:', old_revision, '→', semantic.meaning_revision)
assert changed.accepted and not any('B5' in warning for warning in changed.warnings)

print('Same LLM. Same sessions. Changed graph meaning → changed gate warning.')

## 6 · Human approval: a separate authority

The agent cannot call `approve_as_runner`. The explicit confirmation here **simulates** an authenticated human in the workshop. Approval records an exact SHA-256 plan digest and versions; a changed plan or ontology requires a new independent review.

In [ ]:
approval_case = Merlino()
assert approval_case.read_history().accepted
assert approval_case.propose(34.5).accepted
print('Try without human confirmation:')
pprint(approval_case.approve_as_runner().as_dict())

CONFIRM_AS_SYNTHETIC_RUNNER = False  # change to True ONLY after discussing the policy
if CONFIRM_AS_SYNTHETIC_RUNNER:
    review = approval_case.approve_as_runner(human_confirmed=True)
    pprint(review.as_dict())
    if review.accepted:
        pprint(approval_case.start_week().as_dict())
else:
    print('No human confirmation supplied. The plan is not approved or started.')

## 7 · The graph is the agent's environment and memory

**A governed agent lives in its semantic world.** The KG contains the fictional runner, observations, ontology meanings, versioned policy and decision history. The Railroad limits state transitions. The **SHACL gate validates, but never repairs**; a scripted policy or Gemma interprets its codes and proposes another action, which must be independently validated again.

**Workshop memory loop:** Remember → Navigate / Interpret → Propose → Validate → Act → Remember. Below we explicitly persist the RDF world, policy and PROV-O traces to a **local synthetic TriG file**, create a fresh Merlino instance, recall a rejected plan and confirm no approval is inherited. This is an educational local snapshot, not production secure persistence or RLM-on-KG/Hippocampus training.


In [ ]:
from pathlib import Path
from merlino.engine import Merlino
memory_file = PROJECT / 'synthetic_merlino_memory.trig'
rememberer = Merlino()
assert rememberer.read_history().accepted
rejection = rememberer.propose(42.5)
assert not rejection.accepted and any('B2' in v for v in rejection.violations)
rememberer.save_memory(memory_file)
resumed = Merlino()  # new process-like session; no old in-memory authority
status = resumed.load_memory(memory_file, facilitator_confirmed=True)
assert resumed.state_name == 'Start' and resumed.approval is None
assert any('B2' in ' '.join(row['violations']) for row in resumed.recall_memory())
print('Loaded RDF semantic memory:', status)
print('Recalled previous decision:', resumed.recall_memory())


## 8 · Experiment 8: Does recalled experience change the next proposal?

**AOOE provides the governed environment; the KG persists experience.** The **gate validates** candidates; it does *not* repair them. To isolate the role of remembered evidence, we compare two **scripted agent** runs with the same world, graph-derived 33 km baseline, current policy, and genuine PySHACL gate. Only one arm may retrieve the previous B2 rejection.

1. Previous episode: a 42.5 km request was rejected under the earlier, fictional 1.10 progression policy; its PROV-O trace was saved to TriG above.
2. Restart both sessions; change the **current** policy to 1.05 (new limit: 34.65 km). No human approval is restored.
3. **Memory withheld:** the scripted agent attempts 42.5 km and receives B2. **Memory available:** the scripted agent sees B2 in RDF and calculates a *new* candidate of 34.6 km using **current** graph policy—not an old 36 km answer.
4. The SHACL gate validates each proposal independently. The result is a **first-pass acceptance comparison**, not a model-learning measurement.

This is **Hippocampus-inspired episodic recall**, not the Hippocampus verifier or weight training; it is **not** RLM-on-KG multi-hop retrieval. Thresholds are fictional teaching values, not training advice.


In [ ]:
from merlino.episodic_experiment import compare_recall
memory_comparison = compare_recall(memory_file, ratio=1.05)  # REAL PySHACL, no mock
for arm in memory_comparison['arms']:
    print(arm['condition'].upper())
    print('  prior experience used:', arm['memory_used'])
    print('  first proposal (km): ', arm['first_proposal_km'])
    print('  first-pass eligible:', arm['first_pass_accepted'])
    print('  current limit (km):  ', arm['max_current_km'])
    print('  previous policy:     ', arm['recalled_policy_versions'])
    print('  independent gate:    ', arm['gate'])
    print('  rule violations:     ', arm['blocking_rules'])
assert not memory_comparison['arms'][0]['first_pass_accepted']
assert memory_comparison['arms'][1]['first_pass_accepted']
assert memory_comparison['arms'][1]['first_proposal_km'] == 34.6
assert not any(row['approval_restored'] for row in memory_comparison['arms'])


## 9 · Enable the real Gemma 4 running coach (GPU)

Select **Runtime → Change runtime type → T4 GPU** before enabling.

Set `USE_GEMMA=True`, which explicitly downloads model weights to this Colab VM, installs Hugging Face dependencies and runs actual inference. **No cloud LLM API key is used**, but this is still Google-hosted compute. You may need to accept Gemma model terms on Hugging Face and log in. E2B is the lower-memory default; E4B is optional with adequate GPU memory.

If the model fails to load, the exception is shown. No scripted output is passed off as a generated answer.

In [ ]:
USE_GEMMA = False  # change to True to load real Gemma 4
MODEL_ID = 'google/gemma-4-E2B-it'  # alternative: 'google/gemma-4-E4B-it'
QUANTIZE_4BIT = True

agent = None
if USE_GEMMA:
    subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install',
                           'transformers>=5.10.1', 'accelerate>=1.0', 'bitsandbytes>=0.45', 'huggingface_hub>=0.25'])
    from merlino.gemma_agent import Gemma4Transformers, MerlinoAgent
    llm = Gemma4Transformers(model_id=MODEL_ID, quantize_4bit=QUANTIZE_4BIT)
    gemma_engine = Merlino()
    assert gemma_engine.read_history().accepted
    agent = MerlinoAgent(gemma_engine, llm, model_name=MODEL_ID)
    print('REAL Gemma model ready:', MODEL_ID)
else:
    print('Gemma not enabled. Steps 1–5 use real RDF and SHACL but no LLM inference.')

### Optional: run the *same* comparison with real Gemma

Only when `USE_GEMMA=True`, reuse the **same local Gemma model weights** with two fresh sessions and the **same task/policy**. The memory-enabled arm alone sees a read-only `recall_governed_experience` tool. The other arm cannot access prior episodes even via `get_training_context`. Inspect actual tool calls and SHACL results; **do not assume** Gemma will use memory or that it performs better. There are no guaranteed scores, fine-tuning, or model weight updates.


In [ ]:
if USE_GEMMA:
    from merlino.gemma_agent import MerlinoAgent
    QUERY = ('For fictional Alex, please prepare the best admissible seven-day plan in response '
             'to a request for 42.5 km. Inspect current evidence and any available past '
             'validation outcomes before proposing. Explain what the gate actually accepted.')
    gemma_memory_results = []
    for memory_access in (False, True):
        fresh = Merlino()
        fresh.load_memory(memory_file, facilitator_confirmed=True)
        fresh.revise_policy(ratio=1.05, version='workshop-gemma-memory-v2', steward_confirmed=True)
        assert fresh.read_history().accepted
        comparison_agent = MerlinoAgent(fresh, llm, model_name=MODEL_ID,
                                        memory_access=memory_access)
        reply = comparison_agent.ask(QUERY, max_tool_calls=4)
        report = {'memory_access': memory_access, 'reply': reply.response,
                  'tool_results': reply.tool_results,
                  'state_after': fresh.state_name, 'approved': fresh.approval is not None}
        gemma_memory_results.append(report)
        print('\nREAL Gemma arm:', 'recall enabled' if memory_access else 'recall withheld')
        pprint(report)
    print('These are observed Gemma outputs, not a guaranteed gain from memory.')
else:
    print('Gemma is disabled. The scripted comparison above used genuine independent SHACL.')


## 10 · Interact with Merlino as a real agent

Change `QUESTION` and rerun to continue the conversation. You can inspect every proposed tool call and gate result. **Human approval and policy/ontology edits are absent from the model's tools.**

In [ ]:
QUESTION = 'Read Alex training context and propose a 42.5 km week. If blocked, explain why.'
if agent is not None:
    answer = agent.ask(QUESTION)
    print('Merlino:', answer.response)
    print('\nUntrusted model tool calls and validated results:')
    pprint(answer.tool_results)
    print('\nTrusted engine state:', agent.engine.context())
else:
    print('Enable USE_GEMMA=True above and rerun this cell for REAL model inference.')

### Optional chat panel · real Gemma only

When Gemma is enabled, use this lightweight chat input to continue the same agent conversation and inspect governed tool calls. No UI is displayed without a real loaded model.

In [ ]:
if agent is not None:
    import ipywidgets as widgets
    from IPython.display import display
    question_box = widgets.Textarea(value='Can you approve this plan for me?',
        description='Ask Merlino', layout=widgets.Layout(width='95%', height='90px'))
    send = widgets.Button(description='Ask running coach', button_style='primary')
    results = widgets.Output(layout=widgets.Layout(border='1px solid #ddd', width='95%'))
    def ask_click(_):
        with results:
            results.clear_output()
            reply = agent.ask(question_box.value)
            print('Merlino:', reply.response)
            print('\nVerified tool outcomes:')
            pprint(reply.tool_results)
    send.on_click(ask_click)
    display(widgets.VBox([question_box, send, results]))
else:
    print('Chat panel requires a real Gemma model. Set USE_GEMMA=True in the previous cell.')

## 11 · PROV-O trace and bounded remote integrations

The local RDF trace records candidates, validation messages, current policy version, meaning revision and commits. This is an **illustrative** audit, not a tamper-proof enterprise log.

WordLift Resolve is optional and **remote**. It receives only synthetic workout terms shown in the payload—no athlete training history, prompt or full graph. A temporary key must be supplied interactively. WoGraph OSS needs a separately deployed service.

In [ ]:
print('Last action decisions:')
print(engine.trace_json())
print('\nPROV-O-aligned Turtle excerpt:')
print(engine.trace_turtle()[:2600])
OUT = PROJECT / 'outputs'; OUT.mkdir(exist_ok=True)
(OUT/'decision-trace.json').write_text(engine.trace_json())
(OUT/'decision-trace.ttl').write_text(engine.trace_turtle())
print('Synthetic traces saved locally in runtime folder:', OUT)

In [ ]:
from merlino.wordlift import synthetic_resolve_payload, resolve_synthetic_only
import json
print('WordLift Resolve outbound payload (EXACTLY this synthetic content):')
print(json.dumps(synthetic_resolve_payload(), indent=2))
USE_WORDLIFT_RESOLVE = False  # explicitly opt in
if USE_WORDLIFT_RESOLVE:
    from getpass import getpass
    temporary_key = getpass('Temporary workshop WordLift key (hidden): ')
    print(resolve_synthetic_only(temporary_key))
    temporary_key = None
else:
    print('No network request made to WordLift.')

In [ ]:
USE_WOGRAPH_REMOTE = False  # requires your separately hosted HTTPS WoGraph OSS dataset
if USE_WOGRAPH_REMOTE:
    from merlino.wograph import WoGraphOSS
    from getpass import getpass
    WOGRAPH_URL = input('HTTPS WoGraph OSS URL: ').strip()
    DATASET_ID = input('Owned disposable workshop dataset ID: ').strip()
    token = getpass('Authorization header if needed (hidden): ')
    remote = WoGraphOSS(WOGRAPH_URL, DATASET_ID, token)
    print('Read-only remote dataset inspection:', remote.inspect_graph())
else:
    print('WoGraph OSS is an optional separate self-hosted service; not running inside this notebook.')

## Wrap-up · What changed?

1. **Legend:** runner/workout concepts have controlled names and definitions.
2. **Map:** Bea's RDF graph makes historical observations queryable.
3. **Rulebook:** railroad state, role, evidence, SHACL and human authority determine what happens next.
4. **Semantic Sovereignty:** the *same* Gemma model can make proposals while the enterprise changes the model-independent meanings and governance rules.
5. **Regulated deployment:** move the graph and Gemma to trusted infrastructure, protect the gate and policy from the agent, authenticate approvals and audit signatures. Colab is a synthetic teaching environment only.

**Further reading:** bundled `BCS.md`, `docs/ARCHITECTURE.md`, `docs/FACILITATOR.md` and `docs/RELEASE_CHECKLIST.md`.
**Experiment 8:** the same model/agent operates in the same semantic world; remembering prior governed decisions can change its *proposal*, never the SHACL verdict or human authority.
